In [ ]:
!python --version

In [ ]:
import os
import requests
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Контрольна робота

### 1. ЗАВАНТАЖЕННЯ ФАЙЛУ В СЕСІЮ COLAB

In [ ]:
URL = "https://raw.githubusercontent.com/Egorkud/digital-signals-2026/main/Last_Summer_Ikson_No_Copyright_Music_256KBPS.wav"
FILENAME = "Last_Summer.wav"

if not os.path.exists(FILENAME):
    print("Завантаження аудіофайлу з GitHub...")
    response = requests.get(URL)
    response.raise_for_status() # Перевірка на помилки
    with open(FILENAME, 'wb') as f:
        f.write(response.content)
    print("Файл успішно завантажено!")
else:
    print("Файл вже існує у сесії.")

# Зчитуємо аудіо
fs, data = wavfile.read(FILENAME)

# Перевірка на моно (на випадок, якщо файл все ж стерео)
if data.ndim > 1:
    data = data[:, 0]

# Нормалізація сигналу до діапазону [-1.0, 1.0]
data = data.astype(np.float32)
data /= np.max(np.abs(data))

print(f"Частота дискретизації: {fs} Гц")

### 2. АНІМАЦІЯ СПЕКТРУ СИГНАЛУ

In [ ]:
# Обмежуємо час для анімації (перші 15 секунд)
DURATION_SEC = 15
data_slice = data[:fs * DURATION_SEC]

# Розрахунок STFT (Спектрограми)
# nperseg - розмір вікна. 4096 семплів при 44100Гц дає ~10.7 кадрів на секунду
N_PER_SEG = 4096
frequencies, times, Sxx = spectrogram(data_slice, fs=fs, nperseg=N_PER_SEG, noverlap=0)

# Відфільтруємо частоти, щоб показувати лише до 10 кГц (музичний діапазон)
freq_limit = 10000
freq_mask = frequencies <= freq_limit
f_vis = frequencies[freq_mask]
Sxx_vis = Sxx[freq_mask, :]

# Переводимо амплітуду в логарифмічну шкалу (децибели) для кращого сприйняття
# Додаємо 1e-10 для уникнення логарифму від нуля
Sxx_db = 10 * np.log10(Sxx_vis + 1e-10)

# Обмежуємо мінімальний рівень шуму (-80 дБ)
Sxx_db = np.clip(Sxx_db, a_min=-80, a_max=None)

# --- Налаштування графіки (стиль еквалайзера) ---
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#1e1e1e')  # Темний фон фігури
ax.set_facecolor('#1e1e1e')         # Темний фон графіка

line, = ax.plot(f_vis, Sxx_db[:, 0], color='cyan', lw=2)

ax.set_ylim(-80, 0)
ax.set_xlim(0, freq_limit)
ax.set_title("Динамічний спектр аудіосигналу", color='white', fontsize=14)
ax.set_xlabel("Частота (Гц)", color='white', fontsize=12)
ax.set_ylabel("Амплітуда (дБ)", color='white', fontsize=12)
ax.tick_params(colors='white')
ax.grid(True, color='gray', linestyle=':', alpha=0.5)

# Функція оновлення кадру для анімації
def update(frame_idx):
    line.set_ydata(Sxx_db[:, frame_idx])
    return line,

# Створення анімації
fps = int(fs / N_PER_SEG)
anim = FuncAnimation(
    fig, update,
    frames=len(times),
    interval=1000 / fps,  # Затримка між кадрами в мілісекундах
    blit=True
)

# Закриваємо статичний графік, щоб не дублювався вивід
plt.close(fig)

# Рендер у вбудований віджет HTML5
print("Генерація анімації... Це може зайняти декілька секунд.")
HTML(anim.to_jshtml())